# Lab 2: ice, water and vapour

Follow the [semester's Lab 2 instructions](https://www.uio.no/studier/emner/matnat/fys/FYS2160/h26/obligs_labs/lab2/lab2.pdf), section II A.
Use this notebook to plot the simulation output and keep the final configuration
for the next stage.

## Run the ice, melting and expansion stages

1. Choose **Use as project** to edit `ice2water2vapour.in`.
2. Run the ice stage with the supplied settings and wait for it to finish.
3. Run this notebook, including the final cell, which copies the saved configuration
   into the project files.
4. Edit the input for the next stage using the table below, then run the simulation
   and notebook again. Each stage starts from the preceding stage's final configuration.

| Stage | `Told` (K) | `Tnew` (K) | `runlength` (steps) | Input settings |
| --- | ---: | ---: | ---: | --- |
| Ice | 250 | 251 | 1000 | Keep the supplied NPT settings. |
| Melting and water | 251 | 310 | 10000 | Keep NPT. |
| Expansion | 310 | 311 | 2000 | Enable deformation and NVT; disable NPT. |

For expansion, remove the leading `#` from `fix 2 all deform ...` and
`fix integrate all nvt ...`. Add a leading `#` to `fix 1 all npt ...`.
The deformation expands each box length by a factor of 10.
The timestep is 2 fs, so the three runs last 2, 20 and 4 ps respectively.

**Observe every stage:** take screenshots showing the molecular structures and
arrangements. Look closely at the directions of the oxygen–hydrogen bonds.
Compare with the structures described in the experimental lab text and its appendices.

## Analyse the supplied evaporation run

Download `log_400k.lammps` and `traj_400k.lammpstrj` from the
[Lab 2 data page](https://www.uio.no/studier/emner/matnat/fys/FYS2160/h26/obligs_labs/lab2/data/).
Upload the log into the project files, alongside this notebook, and select it
in the load cell below. Use the supplied trajectory in Ovito to inspect the
vapour and take screenshots. The evaporation stage uses these supplied results;
you do not need to run it in Atomify.

The notebook plots energy and enthalpy for whichever stage you select.
For the first three stages it selects the latest run automatically.


In [ ]:
%pip install lammps-logfile pandas
%matplotlib inline

## Load a stage's output

Run this cell again after each completed simulation. To analyse evaporation,
uncomment the `logfile = Path("log_400k.lammps")` line. Comment it out again
to return to the latest Atomify run. You can also specify an older run's log path.


In [ ]:
from pathlib import Path
import lammps_logfile
import matplotlib

logfile = None  # None selects the latest Atomify run.
# logfile = Path("log_400k.lammps")  # Supplied evaporation data, uploaded to project files.

if logfile is None:
    logs = sorted(Path("runs").glob("*/log_*K.lammps"))
    if not logs:
        raise FileNotFoundError("Finish a simulation first, or select the supplied evaporation log above.")
    logfile = logs[-1]

print("Reading:", logfile)
log = lammps_logfile.File(str(logfile))
print("Log keywords: ", log.get_keywords())


## Plot and interpret

Plot the total energy, enthalpy and temperature below. The input uses LAMMPS
`real` units: temperature is in K, and total energy and enthalpy are in kcal/mol
in LAMMPS's extensive box convention. These energy values are for the whole
simulated system, rather than per water molecule.

For melting and evaporation:

1. How much does the enthalpy change?
2. Which is larger, enthalpy or total energy? Why?
3. Does the simulation reach equilibrium? What in the plots supports your answer?

Compare the phase-transition plots with your experimental observations.
Discuss what the molecular arrangements and energy changes tell you about
entropy and latent heat. Include both plots and structural screenshots in your report.


In [ ]:
s = log.get("Step")
T = log.get("Temp")
E = log.get("TotEng")
H = log.get("Enthalpy")

import matplotlib.pyplot as plt
plt.plot(s, E)
plt.xlabel('Step number')
plt.ylabel('Total energy')
plt.title('Total energy')
plt.show()
plt.plot(s, H)
plt.xlabel('Step number')
plt.ylabel('Enthalpy')
plt.title('Enthalpy')
plt.show()
plt.plot(H, T)
plt.xlabel('Enthalpy')
plt.ylabel('Temperature')
plt.title('Temperature vs Enthalpy')
plt.show()


## Keep the configuration for the next stage

After an Atomify run finishes, this cell copies its `data.<temperature>K` file
from the run folder into the project files, replacing a file of the same name
if present. This preserves positions, velocities and molecular topology.

Then set `Told` to the preceding stage's `Tnew` and choose the next row of the
table above. For expansion, also change the NPT, NVT and deformation lines.
The supplied evaporation recording needs no checkpoint copy.


In [ ]:
import shutil

if logfile.name.lower() == "log_400k.lammps":
    print("Supplied evaporation data: no configuration copy is needed.")
else:
    checkpoint_name = "data." + logfile.stem.removeprefix("log_")
    checkpoint = logfile.parent / checkpoint_name
    if not checkpoint.is_file():
        raise FileNotFoundError(
            f"No {checkpoint_name} in {logfile.parent}. Wait for the run to finish; "
            "if it failed, inspect the simulation console before continuing."
        )
    shutil.copyfile(checkpoint, Path(checkpoint_name))
    print(f"Ready for the next stage: {checkpoint_name}")
